# Access with OpenDAP subsetting

Date: 10/15/2026
Author: Jacob Schaperow
Last Update: JRS, Oct. 15, 2026
Environment: nasa-gesdisc-opendap

In [14]:
import xarray as xr
import earthaccess
import concurrent.futures # for multithreading with xarray.open_dataset
from datetime import datetime
from pydap.net import create_session
import time
import pandas as pd

# to get one year of data with 8 workers (opendap streaming), it took 22 minutes (and two granules failed with HTTP 404)

In [ ]:
# input parameters
shortname = "GPM_3IMERGDF"
Version = "07"
LatMin = 24
LatMax = 34
LonMin = -103
LonMax = -93
bbox = [LonMin, LatMin, LonMax, LatMax]
begin_dt = datetime.strptime("2024-01-01", "%Y-%m-%d") # note: with longer time periods, opendap hangs in current configuration
end_dt = datetime.strptime("2024-01-03", "%Y-%m-%d")
beginyear = begin_dt.year
endyear = end_dt.year

# Create time vector to handle missing data and ensure start and end times are correct
# tt = pd.date_range(start=begin_dt, end=end_dt, freq='1D')

print(
    "Getting data with OPENDAP subsetting for {} version {} from" "{} to {} for a bounding box of ({}, {}, {}, {})".format(
    shortname, Version, begin_dt, end_dt, LatMin, LatMax, LonMin, LonMax)
    )

# get bearer token for EDL authentication (once)
auth = earthaccess.login(strategy="netrc")
bearer_token = auth.token["access_token"]
opendap_session = create_session(session_kwargs = {"token":bearer_token})



Getting data with OPENDAP subsetting for GPM_3IMERGDF version 07 from2024-01-01 00:00:00 to 2024-12-31 00:00:00 for a bounding box of (24, 34, -103, -93)


In [16]:
# Create OpenDAP URLs

# Create search query for OPeNDAP URLs
results = earthaccess.search_data(
    short_name = str(shortname),
    version=str(Version),
    temporal=(begin_dt, end_dt),
    bounding_box=tuple(bbox)
)

# Parse out URL from request, add to OPeNDAP URLs list for querying multiple granules with constraint expressions
opendap_urls = []
for item in results:
    for urls in item['umm']['RelatedUrls']:  # Iterate over RelatedUrls in each request step
        if 'OPENDAP' in urls.get('Description', '').upper():  # Check if 'OPENDAP' is in the Description
            # Extract OPeNDAP URL, replace 
            url = urls['URL'].replace('https', 'dap4')

            # Subset desired variables, along with lat, lon, and time
            # To view all variables, comment out these two lines
            ce = "?dap4.ce=/{}%3B/{}%3B/{}%3B/{}".format("precipitation", "lat", "lon", "time")
            url = url + ce

            # Add URL to list
            opendap_urls.append(url)

print("created list of Opendap URLs")
# print(*opendap_urls, sep="\n") # prints the opendap urls

created list of Opendap URLs


In [17]:
# Stream data with xarray.open_dataset

# Assumes that opendap_urls are in chronological order, which they should be, since they come from earthaccess.search

# Helper function to open the dataset (needed to pass arguments to executor.map)
def fetch_dataset(url, max_retries=3):
    # Try to open the URL up to 'max_retries' times
    for attempt in range(max_retries):
        try:
            return xr.open_dataset(url, engine="pydap", session=opendap_session, chunks={})
        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {url} | Error: {e}")
            time.sleep(2)  # Wait 2 seconds before trying again
    # If it fails max_retries times, print a final message and return None
    print(f"All {max_retries} attempts failed for {url}. File skipped.")
    return None

ds_lst = [] # initialize ds_lst
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    
    # executor.map ensures the result stays in the order of opendap_urls
    results = executor.map(fetch_dataset, opendap_urls)
    ds_lst = [ds for ds in results if ds is not None] # parse results and filter out failures


# At this point, ds_lst is a list of lazily-loaded xarray Datasets backed by Dask arrays

Attempt 1 failed for dap4://opendap.earthdata.nasa.gov/collections/C2723754864-GES_DISC/granules/GPM_3IMERGDF.07%3A3B-DAY.MS.MRG.3IMERG.20240509-S000000-E235959.V07B.nc4?dap4.ce=/precipitation%3B/lat%3B/lon%3B/time | Error: HTTP Error occurred 404 Client Error:  for url: https://opendap.earthdata.nasa.gov/collections/C2723754864-GES_DISC/granules/GPM_3IMERGDF.07%3A3B-DAY.MS.MRG.3IMERG.20240509-S000000-E235959.V07B.nc4.dap?dap4.ce=lat%5B0:1:1799%5D - Failed to fetch data from `http://opendap.earthdata.nasa.gov/collections/C2723754864-GES_DISC/granules/GPM_3IMERGDF.07%3A3B-DAY.MS.MRG.3IMERG.20240509-S000000-E235959.V07B.nc4.dap?dap4.ce=lat[0:1:1799]`
Attempt 1 failed for dap4://opendap.earthdata.nasa.gov/collections/C2723754864-GES_DISC/granules/GPM_3IMERGDF.07%3A3B-DAY.MS.MRG.3IMERG.20241112-S000000-E235959.V07B.nc4?dap4.ce=/precipitation%3B/lat%3B/lon%3B/time | Error: Failed to decode variable 'time': HTTP Error occurred 404 Client Error:  for url: https://opendap.earthdata.nasa.gov/co

In [ ]:
# If we have problems with out-of-order data, we can use this to sort them:
# Sort the list of datasets chronologically based on their ACTUAL internal time
# ds_lst.sort(key=lambda ds: ds.time.isel(time=0).values)

# concatenate to make datacube
if len(ds_lst) > 0:
    print("Concatenating files...")
    ds_concat = xr.concat(
        ds_lst, 
        dim='time',
        coords='minimal', 
        compat='override' 
    )

# Force the dataset to align to tt even if there are missing granules
# ds_concat = ds_concat.reindex(time=tt)

# Ensure datetime global attributes are set correctly
last_dataset = ds_lst[-1] # Grabs the last file chronologically
ds_concat.attrs["EndDate"] = last_dataset.attrs.get("EndDate")
ds_concat.attrs["EndTime"] = last_dataset.attrs.get("EndTime")

print(ds_concat)


Concatenating files...
<xarray.Dataset> Size: 9GB
Dimensions:        (time: 366, lon: 3600, lat: 1800)
Coordinates:
  * lon            (lon) float32 14kB -179.9 -179.9 -179.8 ... 179.8 179.9 179.9
  * time           (time) datetime64[ns] 3kB 2024-01-01 ... 2024-12-31
  * lat            (lat) float64 14kB -89.95 -89.85 -89.75 ... 89.75 89.85 89.95
Data variables:
    precipitation  (time, lon, lat) float32 9GB dask.array<chunksize=(1, 3600, 1800), meta=np.ndarray>
Attributes:
    BeginDate:       2024-01-01
    BeginTime:       00:00:00.000Z
    EndDate:         2024-12-31
    EndTime:         23:59:59.999Z
    FileHeader:      StartGranuleDateTime=2024-01-01T00:00:00.000Z;\nStopGran...
    InputPointer:    3B-HHR.MS.MRG.3IMERG.20240101-S000000-E002959.0000.V07B....
    title:           GPM IMERG Final Precipitation L3 1 day 0.1 degree x 0.1 ...
    DOI:             10.5067/GPM/IMERGDF/DAY/07
    ProductionTime:  2024-06-12T14:19:57.862Z
    created:         2025-02-09T19:27:36Z


In [19]:
# Do spatial subsetting (this happens client-side, I believe. We may want to do it server-side)

# Spatial subsetting
ds_sub = ds_concat.sel(
    lon=slice(bbox[0], bbox[2]),
    lat=slice(bbox[1], bbox[3])
)

print(ds_sub) # now a smaller xarray Dataset (lazy/Dask)

<xarray.Dataset> Size: 15MB
Dimensions:        (time: 366, lon: 100, lat: 100)
Coordinates:
  * lon            (lon) float32 400B -102.9 -102.8 -102.8 ... -93.15 -93.05
  * time           (time) datetime64[ns] 3kB 2024-01-01 ... 2024-12-31
  * lat            (lat) float64 800B 24.05 24.15 24.25 ... 33.75 33.85 33.95
Data variables:
    precipitation  (time, lon, lat) float32 15MB dask.array<chunksize=(1, 100, 100), meta=np.ndarray>
Attributes:
    BeginDate:       2024-01-01
    BeginTime:       00:00:00.000Z
    EndDate:         2024-12-31
    EndTime:         23:59:59.999Z
    FileHeader:      StartGranuleDateTime=2024-01-01T00:00:00.000Z;\nStopGran...
    InputPointer:    3B-HHR.MS.MRG.3IMERG.20240101-S000000-E002959.0000.V07B....
    title:           GPM IMERG Final Precipitation L3 1 day 0.1 degree x 0.1 ...
    DOI:             10.5067/GPM/IMERGDF/DAY/07
    ProductionTime:  2024-06-12T14:19:57.862Z
    created:         2025-02-09T19:27:36Z


In [20]:
# load into memory
ppt = ds_sub.compute()
print(ppt)

<xarray.Dataset> Size: 15MB
Dimensions:        (time: 366, lon: 100, lat: 100)
Coordinates:
  * lon            (lon) float32 400B -102.9 -102.8 -102.8 ... -93.15 -93.05
  * time           (time) datetime64[ns] 3kB 2024-01-01 ... 2024-12-31
  * lat            (lat) float64 800B 24.05 24.15 24.25 ... 33.75 33.85 33.95
Data variables:
    precipitation  (time, lon, lat) float32 15MB 1.145 0.275 0.0 ... 0.0 0.0 0.0
Attributes:
    BeginDate:       2024-01-01
    BeginTime:       00:00:00.000Z
    EndDate:         2024-12-31
    EndTime:         23:59:59.999Z
    FileHeader:      StartGranuleDateTime=2024-01-01T00:00:00.000Z;\nStopGran...
    InputPointer:    3B-HHR.MS.MRG.3IMERG.20240101-S000000-E002959.0000.V07B....
    title:           GPM IMERG Final Precipitation L3 1 day 0.1 degree x 0.1 ...
    DOI:             10.5067/GPM/IMERGDF/DAY/07
    ProductionTime:  2024-06-12T14:19:57.862Z
    created:         2025-02-09T19:27:36Z
